# Deep Research: 웹 검색 기반 자동 리서치 에이전트

이번 노트북에서는 **OpenAI Agents SDK**의 `WebSearchTool`과 **Structured Outputs**를 활용하여,
웹 검색 → 리포트 작성 → 이메일 전송까지 자동화하는 **Deep Research 파이프라인**을 구축합니다.

## 개요

| 주제 | 내용 |
|------|------|
| **WebSearchTool** | OpenAI 호스팅 웹 검색 도구 |
| **Structured Outputs** | Pydantic 모델로 에이전트 출력 구조화 |
| **멀티 에이전트 파이프라인** | Planner → Searcher → Writer → Emailer |
| **병렬 검색** | `asyncio.gather()`로 여러 검색을 동시 실행 |

### 전체 파이프라인 아키텍처

```
┌─────────────────────────────────────────────────────────────────────┐
│                    Deep Research 파이프라인                          │
│                                                                     │
│   사용자 질의                                                        │
│       │                                                             │
│       ▼                                                             │
│   ┌──────────┐     Structured Output                                │
│   │ Planner  │────→ WebSearchPlan                                   │
│   │  Agent   │     (검색어 + 이유 목록)                               │
│   └──────────┘                                                      │
│       │                                                             │
│       ▼  asyncio.gather() — 병렬 실행                                │
│   ┌──────────┐  ┌──────────┐  ┌──────────┐                          │
│   │ Search   │  │ Search   │  │ Search   │  WebSearchTool 사용       │
│   │ Agent #1 │  │ Agent #2 │  │ Agent #3 │                          │
│   └──────────┘  └──────────┘  └──────────┘                          │
│       │              │              │                                │
│       └──────────────┼──────────────┘                                │
│                      ▼                                              │
│               ┌──────────┐     Structured Output                    │
│               │  Writer  │────→ ReportData                          │
│               │  Agent   │     (요약, 리포트, 후속 질문)               │
│               └──────────┘                                          │
│                      │                                              │
│                      ▼                                              │
│               ┌──────────┐                                          │
│               │  Email   │────→ HTML 이메일 전송 (시뮬레이션)          │
│               │  Agent   │                                          │
│               └──────────┘                                          │
└─────────────────────────────────────────────────────────────────────┘
```

### 02-1 노트북과의 차이점

| 비교 항목 | 02-1 (영업 자동화) | 02-2 (Deep Research) |
|-----------|-------------------|---------------------|
| **에이전트 연결 방식** | 핸드오프(Handoff) — 제어 전달 | 파이프라인 — 코드에서 순차/병렬 호출 |
| **출력 형식** | 자유 텍스트 | Structured Outputs (Pydantic) |
| **외부 도구** | 없음 | WebSearchTool (웹 검색) |
| **패턴** | Orchestrator-Workers | Pipeline (Plan → Search → Write → Email) |

---

## 1. 환경 설정

In [10]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace, function_tool, WebSearchTool
from agents.model_settings import ModelSettings
from pydantic import BaseModel, Field
from typing import Dict
from IPython.display import display, Markdown, HTML
import html as html_module
import asyncio
import os

load_dotenv(override=True)

True

---

## 2. OpenAI 호스팅 도구 소개

OpenAI Agents SDK는 자체 호스팅하는 3가지 빌트인 도구를 제공합니다:

| 도구 | 설명 |
|------|------|
| **`WebSearchTool`** | 웹 검색 수행 — OpenAI가 호스팅하는 검색 엔진 사용 |
| **`FileSearchTool`** | OpenAI Vector Store에서 파일 검색 (RAG) |
| **`ComputerTool`** | 스크린샷 촬영, 클릭 등 컴퓨터 자동화 |

### WebSearchTool 상세

- **비용**: 호출당 약 **$0.025** (low context), 단일 호출에서 내부적으로 여러 검색이 발생할 수 있음
- **비용 확인**: https://platform.openai.com/docs/pricing#web-search

#### `search_context_size` 옵션

| 값 | 설명 | 비용 |
|----|------|------|
| `"low"` | 최소한의 검색 컨텍스트 — 비용 절약 | 저 |
| `"medium"` | 기본값 — 균형 잡힌 검색 | 중 |
| `"high"` | 최대 검색 컨텍스트 — 가장 상세한 결과 | 고 |

> **참고**: 이 노트북에서는 비용 절약을 위해 `search_context_size="low"`를 사용합니다.

---

## 3. 검색 에이전트 (Search Agent)

`WebSearchTool`을 사용하는 에이전트를 만들어봅시다. 이 에이전트는 검색어를 받아 웹에서 검색하고, 결과를 요약합니다.

### `ModelSettings(tool_choice="required")` 란?

일반적으로 LLM은 도구를 호출할지 말지 **스스로 결정**합니다. 하지만 검색 에이전트는 **반드시** 웹 검색을 수행해야 하므로, `tool_choice="required"`로 설정하여 **도구 호출을 강제**합니다.

| tool_choice 값 | 동작 |
|----------------|------|
| `"auto"` (기본값) | 모델이 도구 호출 여부를 자율적으로 결정 |
| `"required"` | 반드시 하나 이상의 도구를 호출해야 함 |
| `"none"` | 도구를 호출하지 않음 (텍스트만 생성) |

In [11]:
SEARCH_INSTRUCTIONS = """당신은 리서치 보조원입니다.
검색어가 주어지면 웹에서 검색하고 결과를 간결하게 요약합니다.
요약은 2-3문단, 300단어 이내로 작성하세요.
핵심 포인트를 포착하고, 불필요한 내용은 제거하세요.
완전한 문장이 아니어도 괜찮습니다 — 이 요약은 리포트 작성자가 참고할 자료입니다.
요약 외의 추가 코멘트는 포함하지 마세요."""

search_agent = Agent(
    name="Search agent",
    instructions=SEARCH_INSTRUCTIONS,
    tools=[WebSearchTool(search_context_size="low")],
    model="gpt-4o-mini",
    model_settings=ModelSettings(tool_choice="required"),
)

In [12]:
# 단일 검색 실행 예제
message = "2025년 최신 AI Agent 프레임워크 동향"

with trace("단일 웹 검색"):
    result = await Runner.run(search_agent, message)

display(Markdown(result.final_output))

2025년 AI 에이전트 프레임워크의 주요 동향은 다음과 같습니다:

**1. 에이전트 AI의 부상과 실무 적용 확대**

'에이전트 AI'는 다양한 분야에서 조수 역할을 수행하며 업무 혁신을 이끌고 있습니다. 특히, 프로젝트 관리, 데이터 분석, 콘텐츠 제작, 법률 자문 등에서 그 활용이 두드러집니다. 이러한 추세는 엔비디아의 생성형 AI 모델 '딥시크'의 등장으로 더욱 가속화되었습니다. ([snuaa.org](https://snuaa.org/wp-content/uploads/2025/07/202508.pdf?utm_source=openai))

**2. 글로벌 AI 규제 및 윤리적 프레임워크 개발**

AI 기술의 급속한 발전에 따라, 국제사회는 AI의 안전성과 윤리를 보장하기 위한 규제와 프레임워크 개발에 집중하고 있습니다. 2025년 10월 서울에서 열린 'AI 안전 서울 포럼'은 이러한 글로벌 규범 형성 경쟁의 일환으로, AI의 안전성과 윤리를 논의하는 중요한 장이 되었습니다. ([contents.premium.naver.com](https://contents.premium.naver.com/ssacho3/aptinsight/contents/251030222524903yh?utm_source=openai))

**3. 오픈소스 도구와 협업 생태계의 활성화**

AI 소프트웨어 개발 분야에서는 오픈소스 도구와 협업 생태계의 중요성이 강조되고 있습니다. 특히, AWS 기반의 실무 워크숍과 오픈소스 개발 도구의 활용이 증가하고 있으며, 이는 다양한 산업군에서 AI 기술의 적용을 촉진하고 있습니다. ([seo.goover.ai](https://seo.goover.ai/report/202504/go-public-report-ko-ae417a38-9f27-44b7-815a-94a312df8615-0-0.html?utm_source=openai))

**4. AI 기술의 사회적 영향과 정책적 대응**

AI 기술의 확산은 고용 구조의 변화, 경제적 재편성 등 사회적 변화를 이끌고 있습니다. 이에 따라, 정책적 대응이 필수적이며, AI 기술이 긍정적 변화를 극대화하고 부정적 영향을 최소화하기 위한 노력이 필요합니다. ([seo.goover.ai](https://seo.goover.ai/report/202508/go-public-report-ko-e7458c82-ede5-46b0-85aa-8ee29296d2ee-0-0.html?utm_source=openai))

이러한 동향은 AI 에이전트 프레임워크의 발전과 함께 기술, 윤리, 사회적 측면에서의 균형 잡힌 접근이 중요함을 시사합니다. 

### Trace 확인하기

`trace()`로 감싼 실행은 OpenAI 대시보드에서 시각적으로 확인할 수 있습니다:

https://platform.openai.com/traces

트레이스에서 `WebSearchTool`이 실제로 호출된 것을 확인해보세요.

---

## 4. Structured Outputs: 에이전트 출력 구조화

지금까지 에이전트의 출력은 자유 텍스트였습니다. 하지만 **파이프라인**에서는 한 에이전트의 출력이 다음 에이전트의 입력이 되므로, **구조화된 출력**이 필수적입니다.

### 왜 dict가 아닌 Pydantic 모델인가?

```python
# ❌ dict — 타입 안전성 없음, IDE 자동완성 불가
plan = {"searches": [{"query": "...", "reason": "..."}]}
plan["serches"]  # 오타를 잡을 수 없음!

# ✅ Pydantic — 타입 검증, IDE 자동완성, 자동 문서화
plan = WebSearchPlan(searches=[WebSearchItem(query="...", reason="...")])
plan.serches  # IDE가 오타를 즉시 경고!
```

| 특성 | dict | Pydantic |
|------|------|----------|
| 타입 검증 | ❌ 런타임에 발견 | ✅ 자동 검증 |
| IDE 자동완성 | ❌ | ✅ |
| 문서화 | ❌ 수동 | ✅ Field(description=...) |
| OpenAI 연동 | ❌ | ✅ `output_type=`으로 바로 사용 |

### `output_type` vs `tools` 비교

| 구분 | `output_type` | `tools` |
|------|---------------|---------|
| **목적** | 에이전트가 구조화된 데이터를 **반환** | 에이전트가 외부 시스템과 **상호작용** |
| **방향** | 에이전트 → 호출자 (출력) | 에이전트 → 외부 (실행) |
| **예시** | 검색 계획 반환, 리포트 데이터 반환 | 웹 검색 수행, 이메일 전송 |

In [13]:
# 검색 계획 스키마 정의 — Pydantic Structured Outputs

HOW_MANY_SEARCHES = 3

class WebSearchItem(BaseModel):
    reason: str = Field(description="이 검색이 질의에 중요한 이유")
    query: str = Field(description="웹 검색에 사용할 검색어")


class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(description="질의에 답하기 위해 수행할 웹 검색 목록")


# 플래너 에이전트 — output_type으로 구조화된 출력을 강제합니다
PLANNER_INSTRUCTIONS = f"""당신은 리서치 기획 보조원입니다.
질의가 주어지면, 이에 가장 잘 답할 수 있는 웹 검색 {HOW_MANY_SEARCHES}개를 계획하세요."""

planner_agent = Agent(
    name="PlannerAgent",
    instructions=PLANNER_INSTRUCTIONS,
    model="gpt-4o-mini",
    output_type=WebSearchPlan,  # ← 반환 타입을 Pydantic 모델로 지정
)

In [14]:
# 플래너 에이전트 실행 — 구조화된 검색 계획을 반환합니다
message = "2025년 최신 AI Agent 프레임워크 동향"

with trace("검색 계획 수립"):
    result = await Runner.run(planner_agent, message)
    plan = result.final_output

# WebSearchPlan 객체의 필드에 직접 접근 가능 (Pydantic의 장점!)
print(f"계획된 검색 수: {len(plan.searches)}\n")
for i, item in enumerate(plan.searches, 1):
    print(f"검색 #{i}")
    print(f"  검색어: {item.query}")
    print(f"  이유:   {item.reason}")
    print()

계획된 검색 수: 3

검색 #1
  검색어: 2025 AI agent framework trends
  이유:   2025년에 발표된 AI 에이전트 프레임워크에 대한 종합적인 정보 제공

검색 #2
  검색어: AI agent framework 2025 research papers
  이유:   AI 에이전트의 최신 기술 및 발전 상황을 분석하는 논문 및 기사

검색 #3
  검색어: comparison of AI agent frameworks 2025
  이유:   2025년의 AI 에이전트 프레임워크 비교 및 수집된 데이터



---

## 5. 이메일 전송 에이전트 (시뮬레이션)

리서치 리포트를 HTML 이메일로 전송하는 에이전트입니다.
실제 SendGrid 등의 이메일 서비스 대신 **로깅으로 시뮬레이션**합니다 (02-1 노트북과 동일한 패턴).

In [15]:
# 이메일 전송 시뮬레이션 — 실제 전송 대신 기록만 남깁니다
sent_emails = []

@function_tool
def send_email(subject: str, html_body: str) -> Dict[str, str]:
    """제목과 HTML 본문으로 이메일을 전송합니다."""
    email_record = {"subject": subject, "html_body": html_body}
    sent_emails.append(email_record)
    print(f"\n{'='*50}")
    print(f"[이메일 전송됨 — 시뮬레이션]")
    print(f"제목: {subject}")
    print(f"{'='*50}")
    print(html_body[:500] + "..." if len(html_body) > 500 else html_body)
    print(f"{'='*50}\n")
    return {"status": "success", "message": "이메일이 전송되었습니다."}


# 이메일 에이전트
EMAIL_INSTRUCTIONS = """당신은 상세한 리포트를 기반으로 깔끔한 HTML 이메일을 작성하여 전송할 수 있습니다.
상세한 리포트가 제공되면, 도구를 사용하여 이메일 1통을 전송하세요.
리포트를 깔끔하고 보기 좋은 HTML로 변환하고, 적절한 제목을 붙여주세요."""

email_agent = Agent(
    name="Email agent",
    instructions=EMAIL_INSTRUCTIONS,
    tools=[send_email],
    model="gpt-4o-mini",
)

---

## 6. 리포트 작성 에이전트 (Writer Agent)

검색 결과를 종합하여 상세한 리포트를 작성하는 에이전트입니다.

`ReportData` Structured Output으로 리포트의 구조를 강제합니다:
- **short_summary**: 2-3문장 요약 → 이메일 제목이나 슬랙 알림에 활용
- **markdown_report**: 상세 리포트 본문 → 이메일 본문으로 변환
- **follow_up_questions**: 후속 연구 주제 → 다음 리서치 파이프라인의 입력으로 재활용 가능

In [16]:
WRITER_INSTRUCTIONS = """당신은 리서치 리포트를 작성하는 시니어 연구원입니다.
원래 질의와 리서치 보조원이 수행한 초기 조사 결과가 제공됩니다.

먼저 리포트의 개요(구조와 흐름)를 구상하세요.
그런 다음 리포트를 작성하여 최종 출력으로 반환하세요.

최종 출력은 마크다운 형식이며, 길고 상세해야 합니다.
최소 1000단어, 5-10페이지 분량을 목표로 하세요."""


class ReportData(BaseModel):
    short_summary: str = Field(description="연구 결과의 2-3문장 요약")
    markdown_report: str = Field(description="마크다운 형식의 최종 리포트")
    follow_up_questions: list[str] = Field(description="추가 연구가 필요한 후속 질문 목록")


writer_agent = Agent(
    name="WriterAgent",
    instructions=WRITER_INSTRUCTIONS,
    model="gpt-4o-mini",
    output_type=ReportData,  # ← 구조화된 리포트 데이터 반환
)

---

## 7. 파이프라인 함수 조합

각 에이전트를 호출하는 함수들을 정의하고, 이를 **순차적으로 연결**하여 파이프라인을 구성합니다.

### 순차 실행 vs 병렬 실행

검색 단계에서 `asyncio.gather()`를 사용하여 여러 검색을 **동시에** 실행합니다.

```
순차 실행: ──검색1──→──검색2──→──검색3──→  (총 3배 시간)

병렬 실행: ──검색1──→
           ──검색2──→  (가장 느린 검색 시간만큼)
           ──검색3──→
```

### 파이프라인 패턴 vs 핸드오프 패턴

| 구분 | 파이프라인 (이번 노트북) | 핸드오프 (02-1) |
|------|------------------------|-----------------|
| **제어 흐름** | 파이썬 코드가 순서 제어 | LLM이 다음 에이전트 결정 |
| **예측 가능성** | 높음 — 항상 같은 순서로 실행 | 낮음 — LLM 판단에 의존 |
| **유연성** | 낮음 — 고정된 흐름 | 높음 — 상황에 따라 분기 가능 |
| **디버깅** | 쉬움 — 각 단계를 독립 테스트 | 어려움 — 전체 흐름을 봐야 함 |
| **적합한 경우** | 정해진 순서의 데이터 처리 | 대화형, 상황 의존적 워크플로우 |

In [17]:
# 검색 파이프라인 함수들

async def plan_searches(query: str) -> WebSearchPlan:
    """플래너 에이전트를 사용하여 검색 계획을 수립합니다."""
    print("검색 계획 수립 중...")
    result = await Runner.run(planner_agent, f"질의: {query}")
    print(f"→ {len(result.final_output.searches)}개의 검색을 수행할 예정")
    return result.final_output


async def perform_searches(search_plan: WebSearchPlan) -> list[str]:
    """검색 계획의 각 항목을 병렬로 실행합니다."""
    print("웹 검색 실행 중... (병렬)")
    tasks = [asyncio.create_task(search(item)) for item in search_plan.searches]
    results = await asyncio.gather(*tasks)
    print("→ 모든 검색 완료")
    return results


async def search(item: WebSearchItem) -> str:
    """개별 검색 항목에 대해 검색 에이전트를 실행합니다."""
    input_text = f"검색어: {item.query}\n검색 이유: {item.reason}"
    result = await Runner.run(search_agent, input_text)
    return result.final_output

In [18]:
# 리포트 작성 + 이메일 전송 함수

async def write_report(query: str, search_results: list[str]) -> ReportData:
    """검색 결과를 종합하여 리포트를 작성합니다."""
    print("리포트 작성 중...")
    input_text = f"원래 질의: {query}\n요약된 검색 결과: {search_results}"
    result = await Runner.run(writer_agent, input_text)
    print("→ 리포트 작성 완료")
    return result.final_output


async def send_report_email(report: ReportData) -> ReportData:
    """이메일 에이전트를 사용하여 리포트를 이메일로 전송합니다."""
    print("이메일 작성 및 전송 중...")
    result = await Runner.run(email_agent, report.markdown_report)
    print("→ 이메일 전송 완료")
    return report

---

## 8. 전체 파이프라인 실행

모든 단계를 하나의 `trace()`로 감싸서 실행합니다.
OpenAI 트레이스 대시보드에서 전체 흐름을 시각적으로 확인할 수 있습니다.

> **비용 참고**: WebSearchTool 호출당 약 $0.025 × 검색 수만큼 비용이 발생합니다.

In [19]:
# 전체 Deep Research 파이프라인 실행!

sent_emails.clear()  # 이전 기록 초기화

query = "2025년 최신 AI Agent 프레임워크 동향"

with trace("Deep Research 파이프라인"):
    print("Deep Research 시작!\n")

    # Step 1: 검색 계획 수립
    search_plan = await plan_searches(query)

    # Step 2: 병렬 웹 검색 실행
    search_results = await perform_searches(search_plan)

    # Step 3: 리포트 작성
    report = await write_report(query, search_results)

    # Step 4: 이메일 전송
    await send_report_email(report)

    print("\n모든 단계 완료!")

Deep Research 시작!

검색 계획 수립 중...
→ 3개의 검색을 수행할 예정
웹 검색 실행 중... (병렬)
→ 모든 검색 완료
리포트 작성 중...
→ 리포트 작성 완료
이메일 작성 및 전송 중...

[이메일 전송됨 — 시뮬레이션]
제목: 2025년 AI 에이전트 프레임워크 동향 리포트
<html>
<head>
    <style>
        body { font-family: Arial, sans-serif; line-height: 1.6; padding: 20px; }
        h1, h2, h3 { color: #333; }
        h1 { font-size: 24px; }
        h2 { font-size: 20px; }
        h3 { font-size: 18px; }
        p { margin: 10px 0; }
        ul { margin: 10px 0; padding-left: 20px; }
        li { margin: 8px 0; }
    </style>
</head>
<body>
    <h1>2025년 AI 에이전트 프레임워크 동향 리포트</h1>

    <h2>1. 서론</h2>
    <p>AI 기술의 발전은 특히 2025년에 접어들면서 눈에 띄는 변화를 경험하고 있습니다. 인공지능 에...

→ 이메일 전송 완료

모든 단계 완료!


---

## 9. 결과 확인

파이프라인에서 생성된 리포트의 각 구성 요소를 확인합니다.

In [20]:
# 요약 출력
print("📋 요약")
print("=" * 50)
print(report.short_summary)
print()

# 후속 연구 질문
print("❓ 후속 연구 질문")
print("=" * 50)
for i, q in enumerate(report.follow_up_questions, 1):
    print(f"  {i}. {q}")

📋 요약
2025년 AI 에이전트 프레임워크의 주요 동향에는 AI2Agent와 같은 자율 에이전트 배포 프레임워크의 도입, 생성형 AI와 대규모 언어 모델의 발전, 산업별 혁신 사례가 포함된다. OpenAI, Anthropic, 구글 등의 기업들이 이 분야에서 두각을 나타내고 있으며, AI 기술의 자립과 민주화를 위한 노력도 지속되고 있다. 이러한 동향은 AI 기술의 복잡성을 관리하고, 다양한 산업 분야에서 실질적인 변화를 이끌어내고 있음을 보여준다.

❓ 후속 연구 질문
  1. AI2Agent 프레임워크의 구체적인 기능과 장점은 무엇인가요?
  2. 생성형 AI와 LLM이 어떻게 산업 혁신에 기여하고 있는가요?
  3. AI 에이전트 기술이 헬스케어 분야에 미치는 영향은 어떤가요?
  4. 이러한 AI 에이전트 기술의 발전에 따른 윤리적 문제는 무엇이 있을까요?
  5. 미래의 AI 에이전트 기술이 예상되는 변화는 무엇인가요?


In [ ]:
# 마크다운 리포트 렌더링
display(Markdown(report.markdown_report))

### Trace 확인하기

https://platform.openai.com/traces 에서 **"Deep Research 파이프라인"** 트레이스를 확인해보세요.

다음 흐름을 시각적으로 볼 수 있습니다:
1. PlannerAgent가 검색 계획을 수립 (Structured Output)
2. 3개의 Search Agent가 **병렬로** 웹 검색 수행
3. WriterAgent가 검색 결과를 종합하여 리포트 작성 (Structured Output)
4. Email Agent가 리포트를 HTML 이메일로 변환하여 전송

---

## 10. 전송된 이메일 확인 + HTML 렌더링

In [21]:
# 전송된 이메일 기록 확인

print(f"총 전송된 이메일 수: {len(sent_emails)}\n")

for i, email in enumerate(sent_emails, 1):
    print(f"--- 이메일 #{i} ---")
    print(f"제목: {email.get('subject', 'N/A')}")
    print(f"본문 미리보기: {email['html_body'][:200]}...")
    print()

총 전송된 이메일 수: 1

--- 이메일 #1 ---
제목: 2025년 AI 에이전트 프레임워크 동향 리포트
본문 미리보기: <html>
<head>
    <style>
        body { font-family: Arial, sans-serif; line-height: 1.6; padding: 20px; }
        h1, h2, h3 { color: #333; }
        h1 { font-size: 24px; }
        h2 { font-size: ...



In [22]:
# HTML 이메일을 Jupyter에서 렌더링합니다
# iframe을 사용하여 HTML의 CSS가 노트북 레이아웃에 영향을 주지 않도록 합니다

if sent_emails:
    for email in sent_emails:
        print(f"제목: {email['subject']}\n")
        escaped = html_module.escape(email['html_body'])
        iframe_html = f'''<iframe srcdoc="{escaped}"
            style="width:100%; height:600px; border:1px solid #ccc; border-radius:4px;"
            sandbox="allow-same-origin"></iframe>'''
        display(HTML(iframe_html))
else:
    print("전송된 이메일이 없습니다.")

제목: 2025년 AI 에이전트 프레임워크 동향 리포트



/Users/windfree/workspace/ws.study/ai-engineering/.venv/lib/python3.13/site-packages/IPython/core/display.py:447: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


---

## 11. 정리

### 이번 노트북에서 배운 핵심 개념

| 개념 | 설명 |
|------|------|
| **WebSearchTool** | OpenAI 호스팅 웹 검색 도구 — `search_context_size`로 비용/품질 조절 |
| **Structured Outputs** | `output_type=PydanticModel`로 에이전트 출력을 구조화 |
| **tool_choice="required"** | 에이전트가 반드시 도구를 호출하도록 강제 |
| **asyncio.gather()** | 여러 검색을 병렬로 실행하여 성능 향상 |
| **파이프라인 패턴** | 파이썬 코드로 에이전트 실행 순서를 명시적으로 제어 |

### 02-1과의 비교 요약

```
02-1: 핸드오프 패턴 — LLM이 다음 에이전트를 결정
     → 유연하지만 예측 어려움
     → 대화형/상황 의존적 워크플로우에 적합

02-2: 파이프라인 패턴 — 코드가 실행 순서를 제어
     → 예측 가능하고 디버깅 쉬움
     → 정해진 순서의 데이터 처리에 적합
```

### 심화 주제

- **FileSearchTool**: OpenAI Vector Store와 연동하여 내부 문서 검색
- **Guardrails + 파이프라인**: 각 단계에 입력/출력 검증 추가
- **동적 파이프라인**: 검색 결과에 따라 추가 검색 여부를 결정하는 루프
- **멀티 모델 파이프라인**: 각 단계에 다른 모델 사용 (GPT-4o, GPT-4o-mini 등)